# 09 — Benchmark Stato dell'Arte (SOTA) & Scalabilità Computazionale (Ambito eccDNA)

## Guida Rapida alla Lettura del Notebook
Questo notebook confronta le **4 varianti del nostro Framework FCGR ottimizzato** (`src/fcgr.py`) contro la **Baseline originale** e le **3 principali implementazioni dello Stato dell'Arte (SOTA)** in letteratura:

| Categoria | Metodo | Come funziona in breve |
| :--- | :--- | :--- |
| **Baseline** | `1_Baseline_Python_f32` | Doppio ciclo Python sui caratteri + coordinate con la virgola (`float32`, `16 KB/seq` a $k=6$). |
| **SOTA 1** | `2_SOTA_complexCGR_f64` | Libreria Python di riferimento (*Avila Cartes et al.*) che calcola FCGR in `float64` (`32 KB/seq` a $k=6$). |
| **SOTA 2** | `3_SOTA_DictCounter_f32` | Conteggio k-mer via dizionario `Counter` (*stile DeLUCS / Randhawa*) + mappatura CGR sui k-mer distinti. |
| **SOTA 3** | `4_SOTA_PFCGR_3ch_f32` | *Positional FCGR* (*Avila Cartes et al.*): 3 canali `float32` (Frequenza, Posizione Media, Std Posizione, `48 KB/seq`). |
| **Nostra Var. 0** | `5_Ours_Bitwise_u8` | **Drop-in replacement esatto** (`errore = 0.0`): codifica bitwise NumPy + `uint8` (`4 KB/seq`, **-75% RAM**, massimo speedup). |
| **Nostra Var. 1** | `6_Ours_Opt1_RCLog_u8` | Bitwise + `uint8` (`4 KB/seq`, **-75% RAM**) + **Reverse-Complement** (invarianza filamento) + **`log1p`**. |
| **Nostra Var. 2** | `7_Ours_Opt2_MultiScale_3ch_u8` | **Multi-Scala Frattale** a 3 canali ($k=4,5,6$, `12 KB/seq`, **-25% RAM** vs Baseline / **-75%** vs PFCGR) per **abbattere la sparsità**. |
| **Nostra Var. 3** | `8_Ours_Opt3b_EnrichedMS_3ch_u8` | 3 canali bilanciati (`sum=1`, `12 KB/seq`): `Log-Prob` + **Arricchimento Sovra-Casuale `O/E`** (rimuove rumore GC) + **Multi-Scala**. |

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

import importlib.util
if importlib.util.find_spec('complexcgr') is None:
    import subprocess, sys as _sys
    subprocess.run([_sys.executable, '-m', 'pip', 'install', '-q', '--no-deps', 'complexcgr'], check=False)

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_fcgr_siamese')
if not COLAB_ROOT.exists():
    COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
COMPARISON_DIR = PROJECT_ROOT / 'artifacts' / 'final_fcgr_comparison'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
COMPARISON_DIR.mkdir(parents=True, exist_ok=True)

from src.fcgr import (
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_vectorized,
    compute_fcgr_sota_dict,
    compute_fcgr_sota_pfcgr,
    normalize_fcgr_log,
    compute_multiscale_fcgr,
    compute_opt3_enriched_fcgr,
    quantize_fcgr_uint8,
    compute_fcgr_diagnostics,
)

try:
    from complexcgr import FCGR as ComplexCGR_FCGR
    HAS_COMPLEXCGR = True
    print('Libreria SOTA `complexcgr` caricata con successo!')
except ImportError:
    HAS_COMPLEXCGR = False
    print('Nota: `complexcgr` non disponibile; il benchmark proseguirà con le altre varianti SOTA.')

## 1. Benchmark di Scalabilità Temporale, Memoria e Sparsità al variare di $L$ (`500 bp` $\rightarrow$ `500 kb`)
Cronometriamo la **mediana** su più ripetizioni (per evitare rumore di sistema sulla VM) misurando contemporaneamente **tempo (`ms/seq`)**, **memoria (`KB/seq`)** e **sparsità spaziale (`% di pixel vuoti su tutti i canali`)** per tutti gli 8 metodi.

In [ ]:
rng = np.random.default_rng(42)
LENGTHS = [500, 5_000, 30_000, 100_000, 500_000]
LABELS_L = {500: '500 bp (eccDNA med.)', 5_000: '5 kb (eccDNA lungo)', 30_000: '30 kb', 100_000: '100 kb', 500_000: '500 kb'}
TEST_SEQS = {L: ''.join(rng.choice(list('ACGT'), size=L)) for L in LENGTHS}

def bench_method(fn, n_reps: int):
    out = fn()  # warm-up + cattura output per sparsità
    times = []
    for _ in range(n_reps):
        t0 = time.perf_counter()
        fn()
        times.append((time.perf_counter() - t0) * 1000.0)
    diag = compute_fcgr_diagnostics(out)
    return float(np.median(times)), float(diag['spatial_sparsity_pct'])

bench_rows = []
for k_val in [6, 7]:
    ccgr = ComplexCGR_FCGR(k_val) if HAS_COMPLEXCGR else None
    side = 1 << k_val
    kb_1ch_f32 = (side * side * 4) / 1024.0
    kb_1ch_u8 = (side * side * 1) / 1024.0
    kb_3ch_f32 = (3 * side * side * 4) / 1024.0
    kb_3ch_u8 = (3 * side * side * 1) / 1024.0

    for L in LENGTHS:
        seq = TEST_SEQS[L]
        n_fast = 25 if L <= 30_000 else (7 if L <= 100_000 else 3)
        n_slow = 7 if L <= 5_000 else (3 if L <= 30_000 else 1)

        t_b, sp_b = bench_method(lambda: normalize_fcgr(compute_fcgr(seq, k_val, False)[0]), n_slow)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '1_Baseline_Python_f32', 'time_ms': t_b, 'mem_KB': kb_1ch_f32, 'spatial_sparsity_pct': sp_b, 'channels': 1})

        if HAS_COMPLEXCGR:
            t_c, sp_c = bench_method(lambda: ccgr(seq), n_slow)
            bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '2_SOTA_complexCGR_f64', 'time_ms': t_c, 'mem_KB': kb_1ch_f32 * 2, 'spatial_sparsity_pct': sp_c, 'channels': 1})

        t_d, sp_d = bench_method(lambda: normalize_fcgr(compute_fcgr_sota_dict(seq, k_val, False)[0]), n_slow)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '3_SOTA_DictCounter_f32', 'time_ms': t_d, 'mem_KB': kb_1ch_f32, 'spatial_sparsity_pct': sp_d, 'channels': 1})

        t_p, sp_p = bench_method(lambda: compute_fcgr_sota_pfcgr(seq, k_val)[0], n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '4_SOTA_PFCGR_3ch_f32', 'time_ms': t_p, 'mem_KB': kb_3ch_f32, 'spatial_sparsity_pct': sp_p, 'channels': 3})

        t_v, sp_v = bench_method(lambda: quantize_fcgr_uint8(normalize_fcgr(compute_fcgr_vectorized(seq, k_val, False)[0])), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '5_Ours_Bitwise_u8', 'time_ms': t_v, 'mem_KB': kb_1ch_u8, 'spatial_sparsity_pct': sp_v, 'channels': 1})

        t_o1, sp_o1 = bench_method(lambda: quantize_fcgr_uint8(normalize_fcgr_log(compute_fcgr_vectorized(seq, k_val, False, True)[0])), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '6_Ours_Opt1_RCLog_u8', 'time_ms': t_o1, 'mem_KB': kb_1ch_u8, 'spatial_sparsity_pct': sp_o1, 'channels': 1})

        ks_ms = (k_val - 2, k_val - 1, k_val)
        t_o2, sp_o2 = bench_method(lambda: quantize_fcgr_uint8(compute_multiscale_fcgr(seq, ks=ks_ms, target_k=k_val)[0]), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '7_Ours_Opt2_MultiScale_3ch_u8', 'time_ms': t_o2, 'mem_KB': kb_3ch_u8, 'spatial_sparsity_pct': sp_o2, 'channels': 3})

        t_o3, sp_o3 = bench_method(lambda: quantize_fcgr_uint8(compute_opt3_enriched_fcgr(seq, target_k=k_val)[0]), n_fast)
        bench_rows.append({'k': k_val, 'L_bp': L, 'scale': LABELS_L[L], 'method': '8_Ours_Opt3b_EnrichedMS_3ch_u8', 'time_ms': t_o3, 'mem_KB': kb_3ch_u8, 'spatial_sparsity_pct': sp_o3, 'channels': 3})

sota_bench_df = pd.DataFrame(bench_rows)
sota_bench_df.to_csv(RESULTS_DIR / 'sota_length_scaling_benchmark.csv', index=False)

print('=== 1A. TEMPI MEDIANI (ms / sequenza) A k=6 ===')
display(sota_bench_df[sota_bench_df['k'] == 6].pivot(index='method', columns='scale', values='time_ms')[list(LABELS_L.values())].round(3))

print('\n=== 1B. TEMPI MEDIANI (ms / sequenza) A k=7 ===')
display(sota_bench_df[sota_bench_df['k'] == 7].pivot(index='method', columns='scale', values='time_ms')[list(LABELS_L.values())].round(3))

print('\n=== 1C. CONFRONTO SPARSITA\' SPAZIALE (% PIXEL VUOTI) TRA BASELINE, SOTA E NOSTRE VARIANTI (k=6 vs k=7 @ 500 bp eccDNA) ===')
sp_comp = sota_bench_df[sota_bench_df['L_bp'] == 500].pivot(index='method', columns='k', values='spatial_sparsity_pct').rename(columns={6: 'Sparsità k=6 (%)', 7: 'Sparsità k=7 (%)'})
display(sp_comp.round(2))

## 2. Proiezione Reale sull'Intero Dataset eccDNA (`3.755.079` Sequenze) e su Finestre Lunghe (`30 kb`)
Confronto diretto di **Tempo Totale (minuti / ore)**, **Speedup**, **Memoria RAM (GB)** e **Sparsità (%)**.

In [ ]:
N_FULL_ECCDNA = 3_755_079
N_LONG_WINDOWS = 100_000

for titolo, k_val, L_val, n_seqs in [
    ('Dataset eccDNA Completo (3.75M seq @ 500 bp) — k=6', 6, 500, N_FULL_ECCDNA),
    ('Dataset eccDNA Completo (3.75M seq @ 500 bp) — k=7', 7, 500, N_FULL_ECCDNA),
    ('100.000 Finestre Genomiche Lunghe (30 kb) — k=6', 6, 30_000, N_LONG_WINDOWS),
    ('100.000 Finestre Genomiche Lunghe (30 kb) — k=7', 7, 30_000, N_LONG_WINDOWS),
]:
    sub_df = sota_bench_df[(sota_bench_df['k'] == k_val) & (sota_bench_df['L_bp'] == L_val)].copy()
    base_t = float(sub_df.loc[sub_df['method'] == '1_Baseline_Python_f32', 'time_ms'].iloc[0])
    base_mem = float(sub_df.loc[sub_df['method'] == '1_Baseline_Python_f32', 'mem_KB'].iloc[0])
    sub_df['speedup_x'] = (base_t / sub_df['time_ms']).round(1)
    sub_df['total_minutes'] = ((sub_df['time_ms'] * n_seqs) / 60_000.0).round(2)
    sub_df['total_hours'] = (sub_df['total_minutes'] / 60.0).round(2)
    sub_df['total_RAM_GB'] = ((sub_df['mem_KB'] * n_seqs) / (1024 ** 2)).round(2)
    sub_df['ram_saving_pct'] = ((1.0 - sub_df['mem_KB'] / base_mem) * 100.0).round(1)
    print(f'=== {titolo} ===')
    display(sub_df[['method', 'time_ms', 'speedup_x', 'total_minutes', 'total_hours', 'mem_KB', 'total_RAM_GB', 'ram_saving_pct', 'spatial_sparsity_pct']])

fig, axes = plt.subplots(1, 3, figsize=(21, 5.2))
ecc_k6 = sota_bench_df[(sota_bench_df['k'] == 6) & (sota_bench_df['L_bp'] == 500)].copy()
ecc_k6['minutes'] = (ecc_k6['time_ms'] * N_FULL_ECCDNA) / 60_000.0
ecc_k6['ram_gb'] = (ecc_k6['mem_KB'] * N_FULL_ECCDNA) / (1024 ** 2)
colors = ['#C53030' if 'Baseline' in m else ('#DD6B20' if 'SOTA' in m else '#2F855A') for m in ecc_k6['method']]

axes[0].barh(ecc_k6['method'], ecc_k6['minutes'], color=colors)
axes[0].set_xscale('log')
axes[0].set_xlabel('Minuti per 3.75M eccDNA (k=6) — Scala Log')
axes[0].set_title('Tempo Generazione Intero Dataset eccDNA (k=6)')
for idx_r, (_, row) in enumerate(ecc_k6.iterrows()):
    axes[0].text(row['minutes'] * 1.12, idx_r, f"{row['minutes']:.1f} min", va='center', fontsize=8.5, fontweight='bold')
axes[0].grid(axis='x', alpha=0.3)

axes[1].barh(ecc_k6['method'], ecc_k6['ram_gb'], color=colors)
axes[1].set_xlabel('Memoria RAM / Cache per 3.75M eccDNA (GB, k=6)')
axes[1].set_title('Impronta di Memoria RAM (k=6)')
for idx_r, (_, row) in enumerate(ecc_k6.iterrows()):
    axes[1].text(row['ram_gb'] + 1.5, idx_r, f"{row['ram_gb']:.1f} GB", va='center', fontsize=8.5, fontweight='bold')
axes[1].set_xlim(0, ecc_k6['ram_gb'].max() * 1.25)
axes[1].grid(axis='x', alpha=0.3)

axes[2].barh(ecc_k6['method'], ecc_k6['spatial_sparsity_pct'], color=colors)
axes[2].set_xlabel('Sparsità Spaziale (% Pixel Vuoti su 500 bp, k=6)')
axes[2].set_title('Confronto Sparsità: Baseline vs SOTA vs Nostre')
for idx_r, (_, row) in enumerate(ecc_k6.iterrows()):
    axes[2].text(row['spatial_sparsity_pct'] + 1.2, idx_r, f"{row['spatial_sparsity_pct']:.1f}%", va='center', fontsize=8.5, fontweight='bold')
axes[2].set_xlim(0, 108)
axes[2].grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'sota_full_comparison_eccdna.png', dpi=300, bbox_inches='tight')
plt.show()

## 3. Transizione della Sparsità al variare di $k$ (`4..10`) e Lunghezza $L$ (`500 bp` $\rightarrow$ `500 kb`)
Mostra come la lunghezza $L$ e l'ordine $k$ interagiscono: a $k=6$ una sequenza corta (`500 bp`) riempie solo ~12% delle `4.096` celle, mentre una sequenza lunga (`30 kb`) riempie quasi il 100% delle celle!

In [ ]:
sp_rows = []
for k_val in range(4, 11):
    total_cells = (1 << k_val) ** 2
    for L in LENGTHS:
        _, _, distinct = compute_fcgr_vectorized(TEST_SEQS[L], k_val, return_distinct=True)
        sparsity_pct = (1.0 - distinct / float(total_cells)) * 100.0
        sp_rows.append({'k': k_val, 'scale': LABELS_L[L], 'L_bp': L, 'sparsity_pct': round(sparsity_pct, 2)})

sp_df = pd.DataFrame(sp_rows)
sp_piv = sp_df.pivot(index='k', columns='scale', values='sparsity_pct')[list(LABELS_L.values())]
sp_piv.to_csv(RESULTS_DIR / 'sparsity_by_k_and_length_sota.csv')
display(sp_piv)

fig, ax = plt.subplots(figsize=(10, 5))
for L in LENGTHS:
    sub = sp_df[sp_df['L_bp'] == L]
    lw = 2.8 if L in (500, 30_000) else 1.8
    ax.plot(sub['k'], sub['sparsity_pct'], marker='o', linewidth=lw, label=LABELS_L[L])
ax.axhline(90.0, linestyle='--', color='gray', alpha=0.6, label='Soglia Critica 90% Sparsità')
ax.set_xlabel('Ordine k-mer (k)')
ax.set_ylabel('Sparsità (% Celle Vuote)')
ax.set_title('Sparsità della FCGR Classica al variare di k e della Lunghezza L (bp)')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(COMPARISON_DIR / 'sparsity_regime_by_length.png', dpi=300, bbox_inches='tight')
plt.show()